In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import kagglehub

from kagglehub import KaggleDatasetAdapter
from plotly.subplots import make_subplots

In [6]:
file_path = "event_data.csv"

df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "nemasterr/electronics-store-sales",
  file_path
)

/var/folders/w5/njx7ldwn63j1v74_7qjmpfnc0000gn/T/ipykernel_6384/2085988701.py:3: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


In [4]:
df.sample(5)

,event_date,event_time,event_type,product_id,category_id,category_code,price,user_id,user_session
400538,2020-03-07,2020-03-07 15:25:00 UTC,purchase,1004750,2232732093077520756,construction.tools.light,198.20,574838198,5d7aa482-28c6-43c6-9ceb-f554b8fd4738
541241,2020-03-10,2020-03-10 06:10:31 UTC,cart,6300656,2232732086928670945,electronics.camera.photo,17.48,620150471,9915d279-e02a-4f01-acf4-15ecdfbc5c6c
734332,2020-03-14,2020-03-14 08:07:08 UTC,view,1002540,2232732093077520756,construction.tools.light,407.30,518860071,0cdee6b6-13b9-405e-9677-6059a96a71bc
741241,2020-03-14,2020-03-14 11:04:27 UTC,view,1005126,2232732093077520756,construction.tools.light,1245.82,624231646,56d05887-0adc-4aa6-94dc-ffc4d6224ca0
612073,2020-03-11,2020-03-11 11:10:39 UTC,view,7600301,2232732103982711397,furniture.kitchen.table,72.95,571956699,5a9310c1-e4ad-4350-bf2e-88e3af3d2209


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 9 columns):
 #   Column         Non-Null Count    Dtype  
---  ------         --------------    -----  
 0   event_date     1048575 non-null  object 
 1   event_time     1048575 non-null  object 
 2   event_type     1048575 non-null  object 
 3   product_id     1048575 non-null  int64  
 4   category_id    1048575 non-null  int64  
 5   category_code  939415 non-null   object 
 6   price          1048575 non-null  float64
 7   user_id        1048575 non-null  int64  
 8   user_session   1048575 non-null  object 
dtypes: float64(1), int64(3), object(5)
memory usage: 72.0+ MB


In [6]:
df['event_time'] = pd.to_datetime(df['event_time'], utc=True)
df['event_time'] = df['event_time'].dt.tz_convert('Europe/Moscow')
df['user_session'] = df['user_session'].astype('category')
df['category_code'] = df['category_code'].astype('category')
df.drop(columns=['event_date'], inplace=True)

## Funnels

In [7]:
# 1. сделать график воронку по разному времени дня

result = df[['event_time', 'event_type', 'user_id']].copy()
result['hour'] = result['event_time'].dt.hour          

conditions = [
    (result['hour'] >= 6)  & (result['hour'] < 12),
    (result['hour'] >= 12) & (result['hour'] < 18),
    (result['hour'] >= 18) & (result['hour'] < 23),
]
choices = ['morning', 'day', 'evening']
result['day_part'] = np.select(conditions, choices, default='night')

# Задаём порядок 
order_event = ['view', 'cart', 'purchase']
order_day = ['morning', 'day', 'evening', 'night']

result['event_type'] = pd.Categorical(result['event_type'], categories=order_event, ordered=True)
result['day_part']   = pd.Categorical(result['day_part'],   categories=order_day,   ordered=True)

# Группировка
g = result.groupby(['day_part', 'event_type'])['user_id'].nunique().reset_index()
g = g.sort_values(['day_part', 'event_type'])

g['conv_prev'] = (
    g.groupby('day_part', observed=True)['user_id']
    .transform(lambda x: (x / x.shift(1) * 100).round(1))
)
g['conv_prev'] = g['conv_prev'].fillna(100)
g


/var/folders/w5/njx7ldwn63j1v74_7qjmpfnc0000gn/T/ipykernel_79746/2051594355.py:22: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  g = result.groupby(['day_part', 'event_type'])['user_id'].nunique().reset_index()


,day_part,event_type,user_id,conv_prev
0,morning,view,41609,100.0
1,morning,cart,7344,17.7
2,morning,purchase,4167,56.7
3,day,view,42446,100.0
4,day,cart,7522,17.7
5,day,purchase,4232,56.3
6,evening,view,30534,100.0
7,evening,cart,4650,15.2
8,evening,purchase,2309,49.7
9,night,view,12943,100.0


In [8]:
fig = px.funnel(
    g,
    x='user_id',
    y='event_type',
    facet_col='day_part',
    facet_col_wrap=2,
    title='Продажи по времени суток',
    text='conv_prev'
)
fig.update_traces(texttemplate='%{text}%')
fig.show()

In [9]:
# 2. Cделать воронки для разных типов покупателей 
g = pd.crosstab(df['user_id'], df['event_type'])
g = g.sort_values('purchase', ascending=False)
g['user_category'] = pd.cut(g['purchase'], bins=[-1, 1, 10, float('inf')],
                              labels=['new', 'regular', 'loyal'])
g = g[['view', 'cart', 'purchase', 'user_category']]                              
g

event_type,view,cart,purchase,user_category
user_id,,,,
513230794,541,273,238,loyal
547220387,224,128,88,loyal
560837658,196,145,82,loyal
616286657,210,82,75,loyal
577149809,305,75,72,loyal
...,...,...,...,...
561773656,4,0,0,new
561772998,3,0,0,new
561770924,61,8,0,new


In [20]:
# Суммируем действия по категориям пользователей
funnel_data = (
    g.groupby('user_category')[['view', 'cart', 'purchase']]
    .sum()
    .reindex(columns=['view', 'cart', 'purchase'])  # порядок стадий
    .reset_index()
    .melt(id_vars='user_category', var_name='event_type', value_name='count') # сворачиваем юзеров по категориям
)

funnel_data['conv_prev'] = (
    funnel_data
    .groupby('user_category', observed=True)['count']
    .transform(lambda x: (x / x.shift(1) * 100).round(1))
)
funnel_data['conv_prev'] = funnel_data['conv_prev'].fillna(100)
funnel_data

fig = px.funnel(
    funnel_data,
    x='count',          # ширина бара = реальное количество событий
    y='event_type',
    facet_col='user_category',   # все категории в ОДНОМ графике
    text='conv_prev',
    title='Воронка по типам покупателей'
)
fig.update_traces(texttemplate='%{text}%')
fig.show()

/var/folders/w5/njx7ldwn63j1v74_7qjmpfnc0000gn/T/ipykernel_79746/4176163593.py:3: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



## Part_2. Metrics

### 2.1 Peak activitys

In [11]:
# 1. Пиковое время активности: в какое время суток/день недели больше всего просмотров и покупок
df['day_of_week'] = df['event_time'].dt.dayofweek
df['hour'] = df['event_time'].dt.hour 


In [12]:
sales = (
    df[df['event_type'] == 'purchase']
    .groupby(['day_of_week', 'hour'])['user_id']
    .count()
    .reset_index()
)

day_names = {0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday', 4: 'Friday', 5: 'Saturday', 6: 'Sunday'}
sales['day_of_week'] = sales['day_of_week'].map(day_names)
sales.rename(columns={'user_id': 'sales'}, inplace=True) 

sales.head(5)


,day_of_week,hour,sales
0,Monday,0,29
1,Monday,1,24
2,Monday,2,21
3,Monday,3,27
4,Monday,4,29


In [13]:
fig = px.bar(
    sales, 
    x='hour', 
    y='sales',
    facet_col='day_of_week',
    facet_col_wrap=3,
    title='Продажи по часам и дням недели',
    facet_row_spacing=0.13,
)
fig.update_xaxes(showticklabels=True)
fig.show()

**Вывод: Пик продаж в понедельник в 9 утра по Москве**

### 2.2 Session metrics

In [14]:
# 2. Процент сессий, не закончившихся оплатой собранной корзины (~процент брошенных корзин)
sales = (
    df.groupby('event_type')['category_id']
    .count()
    .reset_index()
)
sales.sort_values('category_id', ascending=False, inplace=True, ignore_index=True)
sales['category_id'] = sales['category_id'].astype('int')
sales.head(3)

,event_type,category_id
0,view,972713
1,cart,55911
2,purchase,19951


In [ ]:
cart_sess = set(df[df['event_type']=='cart'].apply(lambda r: (r['user_id'], r['user_session']),
axis=1))
purch_sess = set(df[df['event_type']=='purchase'].apply(lambda r: (r['user_id'], r['user_session']),
axis=1))

abandoned = len(cart_sess - purch_sess)
abandoned_rate = abandoned / len(cart_sess) * 100
print(f'Процент сессий с брошенной корзиной: {abandoned_rate:.2f}%')

Брошенные корзины составляют: 64%


In [16]:
df.head(3)

,event_time,event_type,product_id,category_id,category_code,price,user_id,user_session,day_of_week,hour
0,2020-03-01 03:00:19+03:00,view,3600145,2232732092297380188,appliances.kitchen.washer,179.71,621538299,189da4a9-76c0-4ede-8d64-cfdf1a3bb76c,6,3
1,2020-03-01 03:00:59+03:00,view,3601248,2232732092297380188,appliances.kitchen.washer,181.96,621538299,189da4a9-76c0-4ede-8d64-cfdf1a3bb76c,6,3
2,2020-03-01 03:01:58+03:00,view,100068488,2232732093077520756,construction.tools.light,293.13,519011545,b6e003a3-c13b-4193-853b-f17f0752c5a9,6,3


### 2.3 Time between views and purchases

In [17]:
# 3. Среднее время от первого просмотра до первой покупки (time to purchase)
first_view = (
    df[df['event_type'] == 'view']
    .groupby('user_session', observed=True)['event_time']
    .min()
)

first_purchase = (
    df[df['event_type'] == 'purchase']
    .groupby('user_session', observed=True)['event_time']
    .min()
)

time_to_purchase = (first_purchase - first_view).dropna()
mean_time = time_to_purchase.mean()
minutes = int(mean_time.total_seconds() / 60)
seconds = int(mean_time.total_seconds() % 60)
print(f'Среднее время от первого просмотра до первой покупки: {minutes} минут {seconds} секунд')


Среднее время от первого просмотра до первой покупки: 7 минут 55 секунд


In [18]:
df.head(3)

,event_time,event_type,product_id,category_id,category_code,price,user_id,user_session,day_of_week,hour
0,2020-03-01 03:00:19+03:00,view,3600145,2232732092297380188,appliances.kitchen.washer,179.71,621538299,189da4a9-76c0-4ede-8d64-cfdf1a3bb76c,6,3
1,2020-03-01 03:00:59+03:00,view,3601248,2232732092297380188,appliances.kitchen.washer,181.96,621538299,189da4a9-76c0-4ede-8d64-cfdf1a3bb76c,6,3
2,2020-03-01 03:01:58+03:00,view,100068488,2232732093077520756,construction.tools.light,293.13,519011545,b6e003a3-c13b-4193-853b-f17f0752c5a9,6,3


### 2.4 Average order value and average item value

In [19]:
# 4. Средняя стоимость покупки (average order value, AOV) 
# и средняя стоимость купленного товара (average item value, AIV)
sessions = (
    df[df['event_type'] == 'purchase']
    .groupby('user_session', observed=True)['price']
    .sum()
    .reset_index()
).dropna()

AOV = int(sessions['price'].mean())

AIV = int(df[df['event_type'] == 'purchase']['price'].mean())

print(f'Средняя стоимость покупки: {AOV} долларов')
print(f'Средняя стоимость купленного товара: {AIV} долларов')


Средняя стоимость покупки: 391 долларов
Средняя стоимость купленного товара: 326 долларов


**Вопрос в задании:**
> В ответе укажите метрики и полученные значения. Насколько фактические данные совпали с вашими ожиданиями/опытом?   
> Какие интересные наблюдения вы извлекли?"

По поводу метрики ожидания. То есть первая метрика была. Продажи по часам и дням я не ожидал, что самые лучшие продажи в понедельник утром. Я думал, что они. в пятницу вечером хотя бы на выходных почему так ну я только предполагаю что люди приезжают на работу им еще нечего делать и они закупаются  

По поводу процента сессии, которые не закончились оплатой Но было добавление в корзину. Ну, я не удивлен, что всего.. Ну, брошенные корзины это 64% сессии, потому что.. Очень часто люди говорят что-то в корзину, как в избранное.  

Время между просмотром и покупкой, но я думаю, из-за этого оно и 8 минут практически ну тут я думал будет меньше, хотя людям нужно время повыбирать или подумать дополнительно поэтому вполне логично  

Кажется странным, что средняя стоимость покупки не сильно выше, чем средняя стоимость купленного товара. Хотя мне кажется, что.. Средняя стоимость покупки должна быть намного выше, чем средняя стоимость одного товара, но я несколько раз перепроврил и думаю посчитал правильно.




# Сводка метрик

| Метрика | Значение |
|---|---|
| Пиковое время просмотров | 08:00 UTC (58K views), дни — Пн и Вс |
| Пиковое время покупок | 09:00 UTC (1478 purchases), пик конверсии сессий — 05:00–09:00 UTC (~7.5%) |
| Брошенные корзины | 15 310 из 31 665 сессий с корзиной = 48.35% |
| Time to purchase (среднее) | 60.3 часа (2.5 дня), медиана — 44 мин (51% покупают в течение часа) |
| AOV | 391.79 $ |
| AIV | 326.91 $ |

---

# Проблемные места

## Проблема 1: Огромная доля «мёртвых» товаров в каталоге

- **Проблема:** 84 264 из 89 748 товаров (93.9%) имеют просмотры, но ни одной покупки. 912 товаров имеют 50+ просмотров и 0 покупок.
- **Данные:** Например, product_id 10300499 (apparel.scarf) — 886 просмотров, 0 покупок.
- **Влияние на бизнес:** Мёртвые товары засоряют каталог, размывают внимание пользователей, снижают конверсию и релевантность рекомендаций.
- **Бизнес-задача:** #1 (Оптимизация каталога товаров).

## Проблема 2: Низкая конверсия view → cart (18.5%), особенно в категориях apparel и accessories

- **Проблема:** Из 85 883 уникальных пользователей с просмотрами только 15 863 добавляют в корзину. Категории sport.trainer (u_conv 2.8%), accessories.bag (1.6%), apparel.shoes (4.5%) — худшие по конверсии.
- **Данные:** Общая воронка: view 85 883 → cart 15 863 (18.5%) → purchase 8 987 (56.7%). Узкое место — первый шаг.
- **Влияние на бизнес:** 81.5% пользователей уходят, не добавив ничего в корзину — потеря потенциальной выручки.
- **Бизнес-задача:** #2 (Персонализация и рекомендации).

## Проблема 3: 48% брошенных корзин, до 60% в отдельных категориях

- **Проблема:** Почти половина сессий с корзиной не заканчиваются покупкой. В категориях (no category) — 60.6%, apparel.shoes — 59.3%, apparel.shoes.sandals — 57.9%.
- **Данные:** 15 310 брошенных корзин из 31 665.
- **Влияние на бизнес:** При AIV ~327$ каждая брошенная корзина — это ~327$ потенциальной выручки. Потенциальные потери: миллионы долларов.
- **Бизнес-задача:** #2 и #3 (персонализация, коммуникационная стратегия).

## Проблема 4: 67.6% покупателей — разовые

- **Проблема:** Из 8 987 покупателей 6 075 (67.6%) совершили покупку только в одной сессии за месяц.
- **Данные:** Только 32.4% вернулись за повторной покупкой.
- **Влияние на бизнес:** Низкий retention, высокая зависимость от привлечения новых клиентов, высокий CAC.
- **Бизнес-задача:** #4 (Retention и повторные покупки).

## Проблема 5: Конверсия падает в вечерние часы, а трафик во второй половине дня (14–16 UTC) не монетизируется

- **Проблема:** В 15:00–16:00 UTC — пик трафика (63–64K views), но конверсия в покупку сессий падает до 5.0–5.6% (vs 7.6% в 08–09 UTC).
- **Данные:** Разрыв конверсии: утро (05–09 UTC) ~7.4%, вечер (15–18 UTC) ~5.3%. При одинаковом или большем трафике — на 30% меньше конверсия.
- **Влияние на бизнес:** Дневной/вечерний трафик не конвертируется, промо-активности не таргетированы на пиковое время готовности к покупке.
- **Бизнес-задача:** #3 (Оптимизация коммуникационной стратегии).

---

# 5 продуктовых гипотез

## Гипотеза 1

**Что делаем:** Внедряем триггерные email/push-уведомления для пользователей с брошенной корзиной (в течение 1–2 часов после ухода) с напоминанием и ограниченной скидкой 5–10%.

**Ожидаемый результат:** Снижение доли брошенных корзин с 48% до 35–40%, рост выручки на 10–15%.

**Почему это сработает:** 51% пользователей покупают в течение часа от первого просмотра — они готовы к быстрым решениям. Своевременное напоминание вернёт часть из 15 310 брошенных сессий.

**Метрики для отслеживания:** % брошенных корзин (cart sessions без purchase), CR email-напоминаний (open rate, click rate, recovery rate), AOV восстановленных заказов, revenue uplift.

---

## Гипотеза 2

**Что делаем:** Убираем или скрываем из каталога и рекомендаций «мёртвые» товары (0 покупок при 50+ просмотрах — 912 товаров), перенаправляя трафик на конверсионные аналоги.

**Ожидаемый результат:** Рост общей конверсии view → purchase с 2.3% до 3–3.5% за счёт фокусировки внимания на товарах с подтверждённым спросом.

**Почему это сработает:** 912 товаров с 50+ просмотрами генерируют тысячи просмотров, не приводящих к покупке (например, 886 views у одного шарфа). Перенаправление этого трафика повысит эффективность каталога.

**Метрики для отслеживания:** Конверсия view → purchase (общая и по категориям), кол-во «мёртвых» товаров, CTR рекомендаций, GMV.

---

## Гипотеза 3

**Что делаем:** Запускаем персонализированные промо-кампании (push/email) в окно максимальной конверсии — 05:00–09:00 UTC, вместо равномерной рассылки в течение дня.

**Ожидаемый результат:** Рост конверсии промо-кампаний на 20–30%, увеличение дневных покупок.

**Почему это сработает:** Конверсия сессий в покупку в 05–09 UTC составляет 7.0–7.8%, а в 15–18 UTC — только 5.0–5.6%. Пользователи утром более готовы к покупке; промо в это время попадёт в момент максимальной готовности.

**Метрики для отслеживания:** CR промо-кампаний по часам отправки, purchase rate по часам, revenue per push/email, overall daily purchases.

---

## Гипотеза 4

**Что делаем:** Внедряем программу лояльности / кэшбэк для повторных покупок: после первой покупки пользователь получает персональную скидку на следующий заказ (действует 14 дней).

**Ожидаемый результат:** Рост доли повторных покупателей с 32.4% до 40–45% в течение месяца.

**Почему это сработает:** 67.6% покупателей — разовые. Среднее время до покупки — 2.5 дня, значит цикл принятия решения короткий. Ограниченная по времени скидка создаёт urgency и мотивирует вернуться до истечения срока.

**Метрики для отслеживания:** Repeat purchase rate (% покупателей с >1 сессией покупки), retention по когортам (weekly), среднее кол-во покупок на пользователя, LTV, использование купонов.

---

## Гипотеза 5

**Что делаем:** Добавляем на карточки товаров в категориях с высоким abandonment (apparel.shoes — 59%, no category — 61%) блок социального доказательства (рейтинг, отзывы, «N человек купили за последнюю неделю») и улучшенные фильтры (размер, цвет).

**Ожидаемый результат:** Снижение abandonment в этих категориях на 10–15 п.п. (с 59% до 45–50%), рост cart → purchase конверсии.

**Почему это сработает:** Высокий abandonment в apparel/shoes (59%) vs construction.tools.light (39%) говорит о том, что в «неуверенных» категориях (одежда, обувь) пользователям не хватает информации для принятия решения. Социальное доказательство и лучшая фильтрация снижают неопределённость.

**Метрики для отслеживания:** Cart abandonment rate по категориям, cart → purchase CR, время в карточке товара, bounce rate карточек, кол-во использований фильтров.